# VGG16: grid search regresi AHI

Satu model menggabungkan seluruh posisi tidur. Scratch, feature extraction (FE), dan fine-tuning langsung (FT) memakai grid yang sama: 12 kombinasi per strategi. FT memakai ImageNet dan head baru, tanpa checkpoint FE. FT membuka block5 mulai features[24]; blok awal tetap beku.

Input, split, dan target memakai preprocessing tersimpan. Acuan: [desain eksperimen](../docs/desain_eksperimen.md) dan [search space](../docs/search_space.md). Test sudah terpapar pada PHASE 5 dan tidak digunakan untuk tuning. Hasil lama tetap tersedia di arsip; notebook ini memakai protokol baru.

In [ ]:
from pathlib import Path
import json
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src' / 'config.py').is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
from IPython.display import display
from src.config import TUNING_DIR, tuning_candidates
from src.tuning import run_grid_search

## Konfigurasi

Flag saat ini dipertahankan pada RUN_TUNING=True, RUN_MODE='full', dan FULL_TRAINING_APPROVED=True sesuai arahan pengguna. Untuk hanya membaca hasil, ubah RUN_TUNING=False atau jalankan sel hasil tersimpan. Run All memanggil training; namespace grid_v1 yang sudah ada ditolak agar hasil tidak tertimpa. Mode smoke memakai LR 1e-3/dropout 0.3 untuk tiga strategi: batch 8, satu batch train dan validation, satu epoch. Smoke hanya diagnostik.

Mode full menguji seluruh grid dengan Adam/MSE, seed 42, maksimal 50 epoch, dan early stopping patience 10. Full training memerlukan persetujuan setelah laporan smoke. Gunakan namespace keluaran baru jika mengulang eksperimen; runner menolak penimpaan.

In [ ]:
ARCHITECTURE = 'VGG16'
RUN_TUNING = True
RUN_MODE = 'full'
FULL_TRAINING_APPROVED = True
OUTPUT_ROOT = TUNING_DIR / 'grid_v1'

display(pd.DataFrame([
    {'learning_rate': config.initial_learning_rate, 'dropout': config.dropout_rates[0]}
    for config in tuning_candidates()
]))

## Pelaksanaan

Runner bersama hanya memuat train/validation, menginisialisasi ulang setiap kandidat, dan menyimpan checkpoint serta diagnostik di direktori tuning. Full training tetap ditolak tanpa flag persetujuan terpisah.

In [ ]:
summary = run_grid_search(
    ARCHITECTURE, output_root=OUTPUT_ROOT, mode=RUN_MODE,
    enabled=RUN_TUNING, full_training_approved=FULL_TRAINING_APPROVED,
)
display(pd.DataFrame([summary]))

## Hasil tersimpan

Sel ini hanya membaca hasil. Metrik smoke bukan hasil penelitian dan tidak dipakai memilih model. Untuk full search, konfigurasi tiap strategi dipilih dari RMSE validation terkecil, lalu MAE jika seri; biaya FT tidak memasukkan FE. Prediksi negatif tetap dipertahankan. Tabel per posisi tersedia untuk kandidat terpilih, termasuk jumlah sampel dan R2 yang dapat tidak terdefinisi.

In [ ]:
result_root = OUTPUT_ROOT / ARCHITECTURE / RUN_MODE
candidate_path = result_root / 'candidates.json'
if candidate_path.is_file():
    records = json.loads(candidate_path.read_text(encoding='utf-8'))
    display(pd.DataFrame(records).reindex(columns=[
        'experiment', 'candidate_id', 'status', 'RMSE', 'MAE', 'MSE',
        'epochs_completed', 'best_epoch', 'training_seconds',
        'optimizer_updates', 'amp_skipped_steps', 'error',
    ]))
    selection_path = result_root / 'selection_validation.json'
    if RUN_MODE == 'full' and selection_path.is_file():
        selection = json.loads(selection_path.read_text(encoding='utf-8'))
        for experiment, record in selection['selected_conditions'].items():
            print(experiment, record['candidate_id'])
            display(pd.read_csv(result_root / record['directory'] / 'metrics_validation.csv'))
else:
    print('Belum ada hasil tuning tersimpan pada namespace ini.')